# TravelMate AI — Giai đoạn 1: Chuẩn bị dữ liệu

Notebook này kiểm tra 1.200 hội thoại đã qua audit template/schema/rule và chia cố định thành Train/Validation/Test. Báo cáo review nằm trong training/reports/dataset_v1_review.json.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/trongnd16092005/travelmate-ai.git"
BRANCH = "feature/ai-itinerary-generation"
REPO_DIR = Path("/content/travelmate-ai")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR / "services" / "ai-service")
print(Path.cwd())

In [ ]:
import sys

SOURCE_DATASET = Path("training/data/travelmate_train_v1.jsonl")
PROCESSED_DIR = Path("training/data/processed")

subprocess.run(
    [
        sys.executable,
        "-m",
        "training.validate_dataset",
        str(SOURCE_DATASET),
        "--minimum-records",
        "1200",
        "--require-metadata",
        "--require-review-status",
        "approved",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "training.prepare_dataset",
        str(SOURCE_DATASET),
        "--output-dir",
        str(PROCESSED_DIR),
        "--seed",
        "42",
    ],
    check=True,
)

In [ ]:
import json

manifest = json.loads((PROCESSED_DIR / "manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest["totalRecords"] == 1200
assert sum(item["records"] for item in manifest["splits"].values()) == 1200
print("\nChuẩn bị dữ liệu thành công.")

## Trước khi train chính thức

Báo cáo audit tự động không thay thế đánh giá độc lập từ người dùng. Sau khi fine-tune, cần đọc thủ công phản hồi của model trên tập Test.